In [ ]:
  # ============================
# Part 0 — الإعداد: ربط Drive + المكتبات + الثوابت الأساسية
# ============================

from google.colab import drive
drive.mount('/content/drive')

import os
import io
import pickle
import copy
import urllib.request
from collections import deque

import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, log_loss
from xgboost import XGBClassifier

!pip install penaltyblog --quiet
import penaltyblog as pb

PROJECT_PATH = '/content/drive/MyDrive/EPL_Predictor_Project'
DATA_RAW_PATH = os.path.join(PROJECT_PATH, 'data', 'raw')
DATA_PROCESSED_PATH = os.path.join(PROJECT_PATH, 'data', 'processed')
MODELS_PATH = os.path.join(PROJECT_PATH, 'models')

# --- ثوابت Elo ---
DEFAULT_ELO = 1500
K_FACTOR = 20
HOME_ADVANTAGE = 100
FORM_WINDOW = 5
MANAGER_DEFAULT_ELO = 1500
MANAGER_XG_ALPHA = 0.15

# --- توحيد أسماء الفرق (أكتر باج تكرر في المشروع كله) ---
master_name_map = {
    'Hull': 'Hull City', 'Ipswich': 'Ipswich Town',
    'Leeds': 'Leeds United', 'Coventry': 'Coventry City',
}

def rename_dict_keys(d, name_map):
    return {name_map.get(k, k): v for k, v in d.items()}

print("✅ Part 0 تمام — Drive متربط، المكتبات جاهزة، الثوابت متعرّفة.")

Mounted at /content/drive
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 951.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.0/58.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 64.1 MB/s eta 0:00:00
✅ Part 0 تمام — Drive متربط، المكتبات جاهزة، الثوابت متعرّفة.


In [ ]:
# ============================
# Part 0b — بيانات المدربين (تاريخ التعيينات + الخبرة الخارجية)
# ============================

manager_appointments = [
    ('Arsenal', 'Unai Emery', '2018-05-23'),
    ('Arsenal', 'Freddie Ljungberg', '2019-11-29'),
    ('Arsenal', 'Mikel Arteta', '2019-12-20'),
    ('Aston Villa', 'Dean Smith', '2018-10-10'),
    ('Aston Villa', 'Steven Gerrard', '2021-11-10'),
    ('Aston Villa', 'Unai Emery', '2022-11-01'),
    ('Bournemouth', 'Jason Tindall', '2020-08-30'),
    ('Bournemouth', 'Jonathan Woodgate', '2021-08-27'),
    ('Bournemouth', 'Scott Parker', '2021-06-23'),
    ('Bournemouth', "Gary O'Neil", '2022-08-30'),
    ('Bournemouth', 'Andoni Iraola', '2023-06-15'),
    ('Bournemouth', 'Marco Rose', '2026-06-01'),
    ('Brentford', 'Thomas Frank', '2018-10-16'),
    ('Brentford', 'Keith Andrews', '2025-06-01'),
    ('Brighton', 'Chris Hughton', '2014-12-30'),
    ('Brighton', 'Graham Potter', '2019-05-20'),
    ('Brighton', 'Roberto De Zerbi', '2022-09-16'),
    ('Brighton', 'Fabian Hurzeler', '2024-06-14'),
    ('Chelsea', 'Maurizio Sarri', '2018-07-14'),
    ('Chelsea', 'Frank Lampard', '2019-07-04'),
    ('Chelsea', 'Thomas Tuchel', '2021-01-26'),
    ('Chelsea', 'Graham Potter', '2022-09-08'),
    ('Chelsea', 'Frank Lampard', '2023-04-06'),
    ('Chelsea', 'Mauricio Pochettino', '2023-07-01'),
    ('Chelsea', 'Enzo Maresca', '2024-06-01'),
    ('Chelsea', 'Liam Rosenior', '2026-01-06'),
    ('Chelsea', 'Calum McFarlane', '2026-04-22'),
    ('Chelsea', 'Xabi Alonso', '2026-07-01'),
    ('Coventry City', 'Frank Lampard', '2024-01-01'),
    ('Crystal Palace', 'Roy Hodgson', '2017-09-12'),
    ('Crystal Palace', 'Patrick Vieira', '2021-06-04'),
    ('Crystal Palace', 'Roy Hodgson', '2023-03-21'),
    ('Crystal Palace', 'Oliver Glasner', '2024-02-19'),
    ('Crystal Palace', 'Pierre Sage', '2026-06-15'),
    ('Everton', 'Marco Silva', '2018-05-31'),
    ('Everton', 'Carlo Ancelotti', '2019-12-21'),
    ('Everton', 'Rafael Benitez', '2021-06-30'),
    ('Everton', 'Frank Lampard', '2022-01-31'),
    ('Everton', 'Sean Dyche', '2023-01-30'),
    ('Everton', 'David Moyes', '2025-01-15'),
    ('Fulham', 'Scott Parker', '2019-02-23'),
    ('Fulham', 'Marco Silva', '2021-07-01'),
    ('Fulham', 'Alvaro Arbeloa', '2026-07-07'),
    ('Hull City', 'Sergej Jakirovic', '2025-06-01'),
    ('Ipswich Town', 'Kieran McKenna', '2021-12-15'),
    ('Ipswich Town', "Gary O'Neil", '2026-06-01'),
    ('Leeds United', 'Marcelo Bielsa', '2018-06-01'),
    ('Leeds United', 'Jesse Marsch', '2022-02-28'),
    ('Leeds United', 'Javi Gracia', '2023-02-21'),
    ('Leeds United', 'Sam Allardyce', '2023-05-03'),
    ('Leeds United', 'Daniel Farke', '2023-07-01'),
    ('Liverpool', 'Jurgen Klopp', '2015-10-08'),
    ('Liverpool', 'Arne Slot', '2024-06-01'),
    ('Liverpool', 'Andoni Iraola', '2026-06-04'),
    ('Man City', 'Pep Guardiola', '2016-07-01'),
    ('Man City', 'Enzo Maresca', '2026-06-29'),
    ('Man United', 'Jose Mourinho', '2016-05-27'),
    ('Man United', 'Ole Gunnar Solskjaer', '2018-12-19'),
    ('Man United', 'Ralf Rangnick', '2021-11-29'),
    ('Man United', 'Erik ten Hag', '2022-05-23'),
    ('Man United', 'Ruben Amorim', '2024-11-11'),
    ('Newcastle', 'Rafael Benitez', '2016-03-16'),
    ('Newcastle', 'Steve Bruce', '2019-07-17'),
    ('Newcastle', 'Eddie Howe', '2021-11-08'),
    ('Newcastle', 'Matthias Jaissle', '2026-08-05'),
    ("Nott'm Forest", 'Steve Cooper', '2021-09-21'),
    ("Nott'm Forest", 'Nuno Espirito Santo', '2023-12-19'),
    ("Nott'm Forest", 'Vitor Pereira', '2024-12-01'),
    ("Nott'm Forest", 'Oliver Glasner', '2026-07-22'),
    ('Sunderland', 'Regis Le Bris', '2024-06-01'),
    ('Tottenham', 'Mauricio Pochettino', '2014-05-27'),
    ('Tottenham', 'Jose Mourinho', '2019-11-20'),
    ('Tottenham', 'Nuno Espirito Santo', '2021-06-30'),
    ('Tottenham', 'Antonio Conte', '2021-11-02'),
    ('Tottenham', 'Ange Postecoglou', '2023-06-06'),
    ('Tottenham', 'Thomas Frank', '2025-06-01'),
    ('West Ham', 'Manuel Pellegrini', '2018-05-23'),
    ('West Ham', 'David Moyes', '2019-12-28'),
    ('West Ham', 'Julen Lopetegui', '2024-05-31'),
    ('Wolves', 'Nuno Espirito Santo', '2017-05-31'),
    ('Wolves', 'Bruno Lage', '2021-06-09'),
    ('Wolves', 'Julen Lopetegui', '2023-01-09'),
    ('Wolves', "Gary O'Neil", '2023-08-14'),
    ('Wolves', 'Vitor Pereira', '2025-01-01'),
]

managers_df = pd.DataFrame(manager_appointments, columns=['Team', 'Manager', 'StartDate']).drop_duplicates()
managers_df['StartDate'] = pd.to_datetime(managers_df['StartDate'])

def get_manager_for_team_at_date(team, match_date, managers_df):
    """بترجع اسم المدرب المسؤول عن الفريق في تاريخ معين، أو None لو مجهول"""
    team_managers = managers_df[managers_df['Team'] == team]
    valid = team_managers[team_managers['StartDate'] <= match_date]
    if len(valid) == 0:
        return None
    return valid.sort_values('StartDate').iloc[-1]['Manager']

external_manager_experience = {
    'Xabi Alonso': {'prior_matches_managed': 180, 'prior_win_rate': 0.66, 'major_trophies': 2, 'top5_league_experience': True},
    'Alvaro Arbeloa': {'prior_matches_managed': 25, 'prior_win_rate': 0.70, 'major_trophies': 0, 'top5_league_experience': True},
    'Matthias Jaissle': {'prior_matches_managed': 200, 'prior_win_rate': 0.65, 'major_trophies': 2, 'top5_league_experience': False},
    'Pierre Sage': {'prior_matches_managed': 100, 'prior_win_rate': 0.55, 'major_trophies': 1, 'top5_league_experience': True},
    'Marco Rose': {'prior_matches_managed': 400, 'prior_win_rate': 0.58, 'major_trophies': 3, 'top5_league_experience': True},
}

def init_manager_elo(manager_name):
    if manager_name in external_manager_experience:
        exp = external_manager_experience[manager_name]
        offset = (exp['prior_win_rate'] - 0.5) * 100 * 4
        return MANAGER_DEFAULT_ELO + offset
    return MANAGER_DEFAULT_ELO

print(f"✅ Part 0b تمام — {len(managers_df)} تعيين مدرب، {len(external_manager_experience)} مدرب بخبرة خارجية.")

✅ Part 0b تمام — 84 تعيين مدرب، 5 مدرب بخبرة خارجية.


In [ ]:
# ============================
# Part 1a — جلب المواسم التاريخية الـ11 (1516 → 2526)
# ============================

seasons = ['1516','1617','1718','1819','1920','2021','2122','2223','2324','2425','2526']
base_url = "https://www.football-data.co.uk/mmz4281/{}/E0.csv"

dfs = []
for s in seasons:
    req = urllib.request.Request(base_url.format(s), headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=15) as response:
        csv_content = response.read().decode('utf-8', errors='replace')
    df_season = pd.read_csv(io.StringIO(csv_content))
    df_season['Season'] = s
    dfs.append(df_season)
    print(f"Season {s}: {df_season.shape[0]} مباراة")

raw = pd.concat(dfs, ignore_index=True)

# اختيار الأعمدة
core_cols = ['Date', 'Season', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR']
odds_cols = [c for c in ['B365H', 'B365D', 'B365A'] if c in raw.columns]
extra_cols = [c for c in ['HS','AS','HST','AST','HC','AC','HY','AY','HR','AR'] if c in raw.columns]
df = raw[core_cols + odds_cols + extra_cols].copy()

# تصحيح صيغ التاريخ المختلطة (قبل أي ترتيب زمني - الباج الشهير)
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, format='mixed')
print(f"\nتواريخ فاشلة (لازم صفر): {df['Date'].isna().sum()}")

# توحيد الأسماء فوراً (قبل أي استخدام)
df['HomeTeam'] = df['HomeTeam'].replace(master_name_map)
df['AwayTeam'] = df['AwayTeam'].replace(master_name_map)

df = df.sort_values('Date').reset_index(drop=True)

print(f"\nإجمالي المباريات التاريخية: {len(df)}")
print(f"أول تاريخ: {df['Date'].min()} — آخر تاريخ: {df['Date'].max()}")

Season 1516: 380 مباراة
Season 1617: 380 مباراة
Season 1718: 380 مباراة
Season 1819: 380 مباراة
Season 1920: 380 مباراة
Season 2021: 380 مباراة
Season 2122: 380 مباراة
Season 2223: 380 مباراة
Season 2324: 380 مباراة
Season 2425: 380 مباراة
Season 2526: 380 مباراة

تواريخ فاشلة (لازم صفر): 0

إجمالي المباريات التاريخية: 4180
أول تاريخ: 2015-08-08 00:00:00 — آخر تاريخ: 2026-05-24 00:00:00


In [ ]:
# ============================
# Part 1b (نهائية) — تحميل xG من الكاش + دمجها في df
# ============================

xg_raw_path = os.path.join(DATA_RAW_PATH, 'understat_xg_raw.csv')
xg_df = pd.read_csv(xg_raw_path)
xg_df['Date'] = pd.to_datetime(xg_df['Date']).dt.normalize()
print(f"عدد صفوف xG المحمّلة من الكاش: {len(xg_df)}")

understat_team_names = {
    'Arsenal': 'Arsenal', 'Aston Villa': 'Aston_Villa', 'Bournemouth': 'Bournemouth',
    'Brentford': 'Brentford', 'Brighton': 'Brighton', 'Chelsea': 'Chelsea',
    'Coventry City': 'Coventry', 'Crystal Palace': 'Crystal_Palace', 'Everton': 'Everton',
    'Fulham': 'Fulham', 'Hull City': 'Hull', 'Ipswich Town': 'Ipswich',
    'Leeds United': 'Leeds', 'Liverpool': 'Liverpool', 'Man City': 'Manchester_City',
    'Man United': 'Manchester_United', 'Newcastle': 'Newcastle_United',
    "Nott'm Forest": 'Nottingham_Forest', 'Sunderland': 'Sunderland', 'Tottenham': 'Tottenham',
}
reverse_name_map = {v.replace('_', ' '): k for k, v in understat_team_names.items()}
reverse_name_map.update({
    'Manchester City': 'Man City', 'Manchester United': 'Man United',
    'Newcastle United': 'Newcastle', 'Nottingham Forest': "Nott'm Forest",
    'West Bromwich Albion': 'West Brom', 'Wolverhampton Wanderers': 'Wolves',
})

xg_df['HomeTeam'] = xg_df['HomeTeam_understat'].map(reverse_name_map).fillna(xg_df['HomeTeam_understat'])
xg_df['AwayTeam'] = xg_df['AwayTeam_understat'].map(reverse_name_map).fillna(xg_df['AwayTeam_understat'])
xg_df['HomeTeam'] = xg_df['HomeTeam'].replace(master_name_map)
xg_df['AwayTeam'] = xg_df['AwayTeam'].replace(master_name_map)

# فحص إلزامي: أسماء في xG مش موجودة في df (المفروض فاضية أو Coventry City بس)
our_teams = set(df['HomeTeam']) | set(df['AwayTeam'])
xg_teams = set(xg_df['HomeTeam']) | set(xg_df['AwayTeam'])
print("أسماء xG غير متطابقة مع df:", xg_teams - our_teams)

xg_merge_data = xg_df[['Date', 'HomeTeam', 'AwayTeam', 'HomeXG', 'AwayXG']].drop_duplicates(subset=['Date', 'HomeTeam', 'AwayTeam'])

df = df.drop(columns=['HomeXG', 'AwayXG', 'XGKnown'], errors='ignore')
n_before = len(df)
df = df.merge(xg_merge_data, on=['Date', 'HomeTeam', 'AwayTeam'], how='left')
assert len(df) == n_before, "الدمج ضاعف صفوف df!"

df['XGKnown'] = df['HomeXG'].notna().astype(int)
df['HomeXG'] = df['HomeXG'].fillna(df['FTHG'])
df['AwayXG'] = df['AwayXG'].fillna(df['FTAG'])

print(f"\nصفوف فيها xG حقيقي: {(df['XGKnown']==1).sum()} (المتوقع: ~3700)")
print(f"صفوف اتعوّضت بالأهداف الفعلية: {(df['XGKnown']==0).sum()} (المتوقع: ~430-480)")

عدد صفوف xG المحمّلة من الكاش: 3753
أسماء xG غير متطابقة مع df: {'Coventry City'}

صفوف فيها xG حقيقي: 3715 (المتوقع: ~3700)
صفوف اتعوّضت بالأهداف الفعلية: 465 (المتوقع: ~430-480)


In [ ]:
# ============================
# Part 1c — جلب نتائج موسم 2026-27 الفعلية وإضافتها لـdf
# ============================

req = urllib.request.Request(
    "https://www.football-data.co.uk/mmz4281/2627/E0.csv",
    headers={'User-Agent': 'Mozilla/5.0'}
)
with urllib.request.urlopen(req, timeout=15) as response:
    csv_content = response.read().decode('utf-8', errors='replace')

df_2627 = pd.read_csv(io.StringIO(csv_content))
df_2627['Date'] = pd.to_datetime(df_2627['Date'], dayfirst=True)
df_2627['Season'] = '2627'

df_2627['HomeTeam'] = df_2627['HomeTeam'].replace(master_name_map)
df_2627['AwayTeam'] = df_2627['AwayTeam'].replace(master_name_map)

odds_cols_2627 = [c for c in ['B365H', 'B365D', 'B365A'] if c in df_2627.columns]
extra_cols_2627 = [c for c in ['HS','AS','HST','AST','HC','AC','HY','AY','HR','AR'] if c in df_2627.columns]

new_season = df_2627[['Date','Season','HomeTeam','AwayTeam','FTHG','FTAG','FTR'] + odds_cols_2627 + extra_cols_2627].copy()

# xG الحقيقي الجاي من المصدر نفسه لموسم 2627 (مش من Understat)
new_season['HomeXG'] = df_2627['HxG'] if 'HxG' in df_2627.columns else np.nan
new_season['AwayXG'] = df_2627['AxG'] if 'AxG' in df_2627.columns else np.nan
new_season['XGKnown'] = new_season['HomeXG'].notna().astype(int)
new_season['HomeXG'] = new_season['HomeXG'].fillna(new_season['FTHG'])
new_season['AwayXG'] = new_season['AwayXG'].fillna(new_season['FTAG'])

df = pd.concat([df, new_season], ignore_index=True, sort=False).sort_values('Date').reset_index(drop=True)

print(f"عدد مباريات موسم 2026-27 المضافة: {len(new_season)}")
print(f"إجمالي صفوف df الآن: {len(df)}")
print(f"آخر مباراة: {df[df['Season']=='2627']['Date'].max()}")

عدد مباريات موسم 2026-27 المضافة: 50
إجمالي صفوف df الآن: 4230
آخر مباراة: 2026-09-20 00:00:00


In [ ]:
# ============================
# Part 1d — تحميل جدول المواعيد الكامل (380 مباراة) من الملف المرفوع
# ============================

fixtures_path = os.path.join(DATA_RAW_PATH, 'epl-2026-GMTStandardTime.csv')
fixtures_raw = pd.read_csv(fixtures_path)

fixtures_df = fixtures_raw.rename(columns={
    'Round Number': 'Round', 'Home Team': 'HomeTeam', 'Away Team': 'AwayTeam'
})[['Round', 'Date', 'HomeTeam', 'AwayTeam']].copy()

fixtures_df['Date'] = pd.to_datetime(fixtures_df['Date'], dayfirst=True)

fixture_name_fix = {**master_name_map, 'Man Utd': 'Man United', 'Spurs': 'Tottenham'}
fixtures_df['HomeTeam'] = fixtures_df['HomeTeam'].replace(fixture_name_fix)
fixtures_df['AwayTeam'] = fixtures_df['AwayTeam'].replace(fixture_name_fix)

print(f"إجمالي المباريات: {len(fixtures_df)} (المتوقع: 380)")
print(f"عدد الجولات: {fixtures_df['Round'].nunique()} (المتوقع: 38)")

team_counts = pd.concat([fixtures_df['HomeTeam'], fixtures_df['AwayTeam']]).value_counts()
print(f"\nكل الفرق 38 مباراة بالظبط؟ {(team_counts == 38).all()}")
if not (team_counts == 38).all():
    print(team_counts[team_counts != 38])

# فحص إلزامي: أسماء الجدول متطابقة مع أسماء df
fixture_teams = set(fixtures_df['HomeTeam']) | set(fixtures_df['AwayTeam'])
df_teams_2627 = set(df[df['Season']=='2627']['HomeTeam']) | set(df[df['Season']=='2627']['AwayTeam'])
print(f"\nأسماء في الجدول مش موجودة في نتائج 2627: {fixture_teams - df_teams_2627}")

إجمالي المباريات: 380 (المتوقع: 380)
عدد الجولات: 38 (المتوقع: 38)

كل الفرق 38 مباراة بالظبط؟ True

أسماء في الجدول مش موجودة في نتائج 2627: set()


In [ ]:
# ============================
# Part 1e — actual_played + remaining_fixtures (فلتر ديناميكي، يحل باج تكرار الجولات)
# ============================

actual_played = df[df['Season']=='2627'][['HomeTeam','AwayTeam','FTHG','FTAG']].rename(
    columns={'FTHG':'HomeGoals','FTAG':'AwayGoals'}
)

dynamic_round_cutoff = len(actual_played) // 10
remaining_fixtures = fixtures_df[fixtures_df['Round'] > dynamic_round_cutoff].copy()

print(f"عدد مباريات actual_played: {len(actual_played)}")
print(f"الجولات المكتملة: {dynamic_round_cutoff}")
print(f"المباريات المتبقية: {len(remaining_fixtures)} (المتوقع: 380 - {len(actual_played)})")
print(f"أول جولة متبقية: {remaining_fixtures['Round'].min()}")

عدد مباريات actual_played: 50
الجولات المكتملة: 5
المباريات المتبقية: 330 (المتوقع: 380 - 50)
أول جولة متبقية: 6


Part 2

In [ ]:
# ============================
# Part 2a — تعيين المدرب المسؤول عن كل مباراة (قبل أي حساب Elo)
# ============================

df['HomeManager'] = df.apply(lambda r: get_manager_for_team_at_date(r['HomeTeam'], r['Date'], managers_df), axis=1)
df['AwayManager'] = df.apply(lambda r: get_manager_for_team_at_date(r['AwayTeam'], r['Date'], managers_df), axis=1)

print(f"✅ Part 2a — مباريات بمدرب معروف للمضيف: {df['HomeManager'].notna().sum()} / {len(df)}")

✅ Part 2a — مباريات بمدرب معروف للمضيف: 2715 / 4230


In [ ]:
# ============================
# Part 2a-ب — دوال Elo الأساسية (كانت ناقصة)
# ============================

def update_elo_ratings(home_rating, away_rating, home_goals, away_goals, k_factor=20, home_advantage=100):
    adjusted_home_rating = home_rating + home_advantage
    expected_home = 1 / (1 + 10 ** ((away_rating - adjusted_home_rating) / 400))
    expected_away = 1 - expected_home

    if home_goals > away_goals:
        actual_home, actual_away = 1.0, 0.0
        winner_rating, loser_rating = adjusted_home_rating, away_rating
    elif home_goals < away_goals:
        actual_home, actual_away = 0.0, 1.0
        winner_rating, loser_rating = away_rating, adjusted_home_rating
    else:
        actual_home, actual_away = 0.5, 0.5
        winner_rating, loser_rating = adjusted_home_rating, away_rating

    goal_diff = abs(home_goals - away_goals)
    if goal_diff == 0:
        multiplier = 1.0
    else:
        rating_diff = winner_rating - loser_rating
        multiplier = np.log(goal_diff + 1) * (2.2 / (rating_diff * 0.001 + 2.2))

    new_home_rating = home_rating + k_factor * multiplier * (actual_home - expected_home)
    new_away_rating = away_rating + k_factor * multiplier * (actual_away - expected_away)
    return new_home_rating, new_away_rating


def get_adaptive_k_factor(matchweek_number, base_k=20, early_season_games=6, min_k_ratio=0.5):
    if matchweek_number > early_season_games:
        return base_k
    ratio = min_k_ratio + (1 - min_k_ratio) * (matchweek_number - 1) / (early_season_games - 1)
    return base_k * ratio

print("✅ دوال Elo جاهزة.")

✅ دوال Elo جاهزة.


In [ ]:
# ============================
# Part 2b — إعادة بناء كل الحالة التاريخية + أعمدة df لكل صف (تمريرة واحدة بس)
# الفيكس الأساسي: K-Factor التكيّفي بيسري على موسم 2627 الحالي بس، مش كل التاريخ
# ============================

elo_ratings = {}
manager_elo = {}
team_points_history = {}
team_goals_scored_history = {}
team_goals_conceded_history = {}
team_xg_scored_history = {}
team_xg_conceded_history = {}
last_match_date_full = {}
relegation_elo_pool = []
matches_played_per_season = {}
previous_season = None

df = df.sort_values('Date').reset_index(drop=True)
season_team_sets = {
    s: set(df[df['Season']==s]['HomeTeam']) | set(df[df['Season']==s]['AwayTeam'])
    for s in df['Season'].unique()
}

cols = {k: [] for k in [
    'HomeElo','AwayElo','EloDiff','EloCloseness',
    'HomeManagerElo','AwayManagerElo','HomeManagerKnown','AwayManagerKnown',
    'HomeFormPoints','HomeFormGoalsScored','HomeFormGoalsConceded',
    'AwayFormPoints','AwayFormGoalsScored','AwayFormGoalsConceded',
    'HomeDaysRest','AwayDaysRest','HomeShortRest','AwayShortRest',
]}

for idx, row in df.iterrows():
    season = row['Season']
    home, away = row['HomeTeam'], row['AwayTeam']
    hg, ag = row['FTHG'], row['FTAG']
    home_mgr, away_mgr = row.get('HomeManager'), row.get('AwayManager')
    match_date = row['Date']

    # --- تهيئة الفرق الجديدة في موسم جديد + تجميع Elo الفرق الهابطة ---
    if season != previous_season:
        current_teams = season_team_sets[season]
        for team in current_teams:
            if team not in elo_ratings:
                init_rating = np.mean(relegation_elo_pool[-3:]) if relegation_elo_pool else DEFAULT_ELO
                elo_ratings[team] = init_rating
        if previous_season is not None:
            prev_teams = season_team_sets[previous_season]
            relegated = prev_teams - current_teams
            for team in relegated:
                relegation_elo_pool.append(elo_ratings[team])
        previous_season = season

    # --- K-Factor: تكيّفي للموسم الحالي بس، ثابت لكل التاريخ (الفيكس) ---
    home_mw = matches_played_per_season.get((season, home), 0) + 1
    away_mw = matches_played_per_season.get((season, away), 0) + 1
    if season == '2627':
        adaptive_k = get_adaptive_k_factor(max(home_mw, away_mw))
    else:
        adaptive_k = K_FACTOR

    # --- Elo قبل المباراة (snapshot) ---
    h_elo = elo_ratings.get(home, DEFAULT_ELO)
    a_elo = elo_ratings.get(away, DEFAULT_ELO)
    cols['HomeElo'].append(h_elo); cols['AwayElo'].append(a_elo)
    cols['EloDiff'].append(h_elo - a_elo); cols['EloCloseness'].append(-abs(h_elo - a_elo))

    # --- Manager Elo قبل المباراة (snapshot) ---
    h_mgr_elo = manager_elo.get(home_mgr, MANAGER_DEFAULT_ELO) if home_mgr else MANAGER_DEFAULT_ELO
    a_mgr_elo = manager_elo.get(away_mgr, MANAGER_DEFAULT_ELO) if away_mgr else MANAGER_DEFAULT_ELO
    cols['HomeManagerElo'].append(h_mgr_elo); cols['AwayManagerElo'].append(a_mgr_elo)
    cols['HomeManagerKnown'].append(int(home_mgr is not None)); cols['AwayManagerKnown'].append(int(away_mgr is not None))

    # --- الفورم قبل المباراة (snapshot) ---
    h_pts_hist = team_points_history.get(home, deque(maxlen=FORM_WINDOW))
    a_pts_hist = team_points_history.get(away, deque(maxlen=FORM_WINDOW))
    h_gs_hist = team_goals_scored_history.get(home, deque(maxlen=FORM_WINDOW))
    h_gc_hist = team_goals_conceded_history.get(home, deque(maxlen=FORM_WINDOW))
    a_gs_hist = team_goals_scored_history.get(away, deque(maxlen=FORM_WINDOW))
    a_gc_hist = team_goals_conceded_history.get(away, deque(maxlen=FORM_WINDOW))

    cols['HomeFormPoints'].append(sum(h_pts_hist) if h_pts_hist else np.nan)
    cols['HomeFormGoalsScored'].append(np.mean(h_gs_hist) if h_gs_hist else np.nan)
    cols['HomeFormGoalsConceded'].append(np.mean(h_gc_hist) if h_gc_hist else np.nan)
    cols['AwayFormPoints'].append(sum(a_pts_hist) if a_pts_hist else np.nan)
    cols['AwayFormGoalsScored'].append(np.mean(a_gs_hist) if a_gs_hist else np.nan)
    cols['AwayFormGoalsConceded'].append(np.mean(a_gc_hist) if a_gc_hist else np.nan)

    # --- أيام الراحة قبل المباراة ---
    h_rest = (match_date - last_match_date_full[home]).days if home in last_match_date_full else np.nan
    a_rest = (match_date - last_match_date_full[away]).days if away in last_match_date_full else np.nan
    h_rest = min(h_rest, 45) if pd.notna(h_rest) else np.nan
    a_rest = min(a_rest, 45) if pd.notna(a_rest) else np.nan
    cols['HomeDaysRest'].append(h_rest); cols['AwayDaysRest'].append(a_rest)
    cols['HomeShortRest'].append(int(h_rest <= 4) if pd.notna(h_rest) else 0)
    cols['AwayShortRest'].append(int(a_rest <= 4) if pd.notna(a_rest) else 0)

    # --- تحديث Elo بعد المباراة ---
    new_h, new_a = update_elo_ratings(h_elo, a_elo, hg, ag, k_factor=adaptive_k, home_advantage=HOME_ADVANTAGE)
    elo_ratings[home], elo_ratings[away] = new_h, new_a

    if home_mgr and away_mgr:
        if home_mgr not in manager_elo: manager_elo[home_mgr] = init_manager_elo(home_mgr)
        if away_mgr not in manager_elo: manager_elo[away_mgr] = init_manager_elo(away_mgr)
        new_hm, new_am = update_elo_ratings(manager_elo[home_mgr], manager_elo[away_mgr], hg, ag,
                                              k_factor=adaptive_k, home_advantage=HOME_ADVANTAGE)
        manager_elo[home_mgr], manager_elo[away_mgr] = new_hm, new_am

    # --- تحديث النقط/الفورم/xG بعد المباراة ---
    if hg > ag: h_pts, a_pts = 3, 0
    elif hg < ag: h_pts, a_pts = 0, 3
    else: h_pts, a_pts = 1, 1

    hxg = row.get('HomeXG', hg) if pd.notna(row.get('HomeXG', np.nan)) else hg
    axg = row.get('AwayXG', ag) if pd.notna(row.get('AwayXG', np.nan)) else ag

    for team_name, pts, gs, gc, xgs, xgc in [(home, h_pts, hg, ag, hxg, axg), (away, a_pts, ag, hg, axg, hxg)]:
        for d in [team_points_history, team_goals_scored_history, team_goals_conceded_history,
                  team_xg_scored_history, team_xg_conceded_history]:
            if team_name not in d:
                d[team_name] = deque(maxlen=FORM_WINDOW)
        team_points_history[team_name].append(pts)
        team_goals_scored_history[team_name].append(gs)
        team_goals_conceded_history[team_name].append(gc)
        team_xg_scored_history[team_name].append(xgs)
        team_xg_conceded_history[team_name].append(xgc)

    last_match_date_full[home] = match_date
    last_match_date_full[away] = match_date
    matches_played_per_season[(season, home)] = home_mw
    matches_played_per_season[(season, away)] = away_mw

for col_name, values in cols.items():
    df[col_name] = values

print("✅ Part 2b — تمت إعادة بناء كل الحالة + كل أعمدة df في تمريرة واحدة.")
print(f"\nتحقق Elo الحالي: Hull City={elo_ratings.get('Hull City'):.1f}, Coventry City={elo_ratings.get('Coventry City'):.1f}, "
      f"Tottenham={elo_ratings.get('Tottenham'):.1f}, Man City={elo_ratings.get('Man City'):.1f}, Arsenal={elo_ratings.get('Arsenal'):.1f}")
print(f"\nقيم مفقودة في الأعمدة الجديدة (أول مباراة لكل فريق متوقع يكون NaN في الفورم/الراحة):")
print(df[list(cols.keys())].isna().sum())

✅ Part 2b — تمت إعادة بناء كل الحالة + كل أعمدة df في تمريرة واحدة.

تحقق Elo الحالي: Hull City=1421.1, Coventry City=1325.2, Tottenham=1461.2, Man City=1743.3, Arsenal=1738.4

قيم مفقودة في الأعمدة الجديدة (أول مباراة لكل فريق متوقع يكون NaN في الفورم/الراحة):
HomeElo                   0
AwayElo                   0
EloDiff                   0
EloCloseness              0
HomeManagerElo            0
AwayManagerElo            0
HomeManagerKnown          0
AwayManagerKnown          0
HomeFormPoints           18
HomeFormGoalsScored      18
HomeFormGoalsConceded    18
AwayFormPoints           17
AwayFormGoalsScored      17
AwayFormGoalsConceded    17
HomeDaysRest             18
AwayDaysRest             17
HomeShortRest             0
AwayShortRest             0
dtype: int64


In [17]:
# ============================
# Part 3 (نهائية - Production) — Dixon-Coles xi=0.001 + XGBoost بالإعدادات الفائزة
# ============================

dc_data = df[['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG']].copy()
dc_data = dc_data.sort_values('Date').reset_index(drop=True)

XI = 0.001
dc_data['weight'] = pb.models.dixon_coles_weights(dc_data['Date'], XI)
dc_model = pb.models.DixonColesGoalModel(
    dc_data['FTHG'], dc_data['FTAG'], dc_data['HomeTeam'], dc_data['AwayTeam'], dc_data['weight']
)
dc_model.fit()
params = dc_model.get_params()

teams_list = sorted(dc_data['HomeTeam'].unique())
attack_dict = rename_dict_keys({t: params[f'attack_{t}'] for t in teams_list}, master_name_map)
defence_dict = rename_dict_keys({t: params[f'defence_{t}'] for t in teams_list}, master_name_map)

df['HomeAttack'] = df['HomeTeam'].map(attack_dict)
df['HomeDefence'] = df['HomeTeam'].map(defence_dict)
df['AwayAttack'] = df['AwayTeam'].map(attack_dict)
df['AwayDefence'] = df['AwayTeam'].map(defence_dict)
df['AttackCloseness'] = -(df['HomeAttack'] - df['AwayAttack']).abs()

feature_cols_simulation = [
    'HomeElo', 'AwayElo', 'EloDiff',
    'HomeFormPoints', 'HomeFormGoalsScored', 'HomeFormGoalsConceded',
    'AwayFormPoints', 'AwayFormGoalsScored', 'AwayFormGoalsConceded',
    'HomeManagerElo', 'AwayManagerElo', 'HomeManagerKnown', 'AwayManagerKnown',
    'HomeDaysRest', 'AwayDaysRest', 'HomeShortRest', 'AwayShortRest',
    'HomeAttack', 'HomeDefence', 'AwayAttack', 'AwayDefence',
    'EloCloseness', 'AttackCloseness',
]

test_seasons = ['2425', '2526']
train_mask = ~df['Season'].isin(test_seasons)
test_mask = df['Season'].isin(test_seasons)

target_map = {'H': 0, 'D': 1, 'A': 2}
y = df['FTR'].map(target_map)
y_train, y_test = y[train_mask], y[test_mask]

X_sim = df[feature_cols_simulation].copy()
X_train, X_test = X_sim[train_mask], X_sim[test_mask]

simulation_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', XGBClassifier(
        n_estimators=150, max_depth=4, learning_rate=0.03,
        subsample=0.7, colsample_bytree=0.7, min_child_weight=1,
        random_state=42, eval_metric='mlogloss'
    ))
])
simulation_model.fit(X_train, y_train)

y_pred_proba = simulation_model.predict_proba(X_test)
acc = accuracy_score(y_test, y_pred_proba.argmax(axis=1))
ll = log_loss(y_test, y_pred_proba)

print(f"Dixon-Coles: Home Advantage={params.get('home_advantage'):.3f}, Rho={params.get('rho'):.3f}")
print(f"\nAccuracy نهائية: {acc:.4f} (Baseline الجديد المعتمد: 0.5066)")
print(f"LogLoss نهائي  : {ll:.4f}")

Dixon-Coles: Home Advantage=0.174, Rho=-0.058

Accuracy نهائية: 0.5066 (Baseline الجديد المعتمد: 0.5066)
LogLoss نهائي  : 1.0131


In [18]:
# ============================
# Part 4a — الثوابت الإضافية + base_state (نقطة انطلاق المحاكاة)
# ============================

current_20_teams_list = sorted(df[df['Season']=='2627']['HomeTeam'].unique())
current_managers = {team: get_manager_for_team_at_date(team, df['Date'].max(), managers_df) for team in current_20_teams_list}

base_state = {
    'elo': elo_ratings,
    'manager_elo': manager_elo,
    'points_hist': team_points_history,
    'gs_hist': team_goals_scored_history,
    'gc_hist': team_goals_conceded_history,
    'last_date': last_match_date_full,
}

print(f"✅ عدد الفرق الحالية: {len(current_20_teams_list)}")
print(f"المدربين الحاليين: {current_managers}")

✅ عدد الفرق الحالية: 20
المدربين الحاليين: {'Arsenal': 'Mikel Arteta', 'Aston Villa': 'Unai Emery', 'Bournemouth': 'Marco Rose', 'Brentford': 'Keith Andrews', 'Brighton': 'Fabian Hurzeler', 'Chelsea': 'Xabi Alonso', 'Coventry City': 'Frank Lampard', 'Crystal Palace': 'Pierre Sage', 'Everton': 'David Moyes', 'Fulham': 'Alvaro Arbeloa', 'Hull City': 'Sergej Jakirovic', 'Ipswich Town': "Gary O'Neil", 'Leeds United': 'Daniel Farke', 'Liverpool': 'Andoni Iraola', 'Man City': 'Enzo Maresca', 'Man United': 'Ruben Amorim', 'Newcastle': 'Matthias Jaissle', "Nott'm Forest": 'Oliver Glasner', 'Sunderland': 'Regis Le Bris', 'Tottenham': 'Thomas Frank'}


In [19]:
# ============================
# Part 4b — محرك المحاكاة (build_simulation_features, predict_match_outcome, update_simulation_state)
# K-Factor تكيّفي مربوط صح من غير ما يسري على التاريخ (نفس فيكس Part 2b)
# ============================

def build_simulation_features(home, away, match_date, sim_state):
    home_mgr = current_managers.get(home)
    away_mgr = current_managers.get(away)

    h_elo = sim_state['elo'].get(home, DEFAULT_ELO)
    a_elo = sim_state['elo'].get(away, DEFAULT_ELO)

    h_days_rest = (match_date - sim_state['last_date'].get(home, match_date - pd.Timedelta(days=7))).days
    a_days_rest = (match_date - sim_state['last_date'].get(away, match_date - pd.Timedelta(days=7))).days
    h_days_rest = min(h_days_rest, 45) if h_days_rest >= 0 else 7
    a_days_rest = min(a_days_rest, 45) if a_days_rest >= 0 else 7

    def form_stats(hist_dict, team):
        hist = hist_dict.get(team, deque(maxlen=FORM_WINDOW))
        return list(hist) if len(hist) > 0 else None

    h_pts = form_stats(sim_state['points_hist'], home)
    a_pts = form_stats(sim_state['points_hist'], away)
    h_gs = form_stats(sim_state['gs_hist'], home)
    h_gc = form_stats(sim_state['gc_hist'], home)
    a_gs = form_stats(sim_state['gs_hist'], away)
    a_gc = form_stats(sim_state['gc_hist'], away)

    h_mgr_elo = sim_state['manager_elo'].get(home_mgr, MANAGER_DEFAULT_ELO) if home_mgr else MANAGER_DEFAULT_ELO
    a_mgr_elo = sim_state['manager_elo'].get(away_mgr, MANAGER_DEFAULT_ELO) if away_mgr else MANAGER_DEFAULT_ELO

    h_attack, h_defence = attack_dict.get(home, np.nan), defence_dict.get(home, np.nan)
    a_attack, a_defence = attack_dict.get(away, np.nan), defence_dict.get(away, np.nan)

    return {
        'HomeElo': h_elo, 'AwayElo': a_elo, 'EloDiff': h_elo - a_elo,
        'HomeFormPoints': sum(h_pts) if h_pts else np.nan,
        'HomeFormGoalsScored': np.mean(h_gs) if h_gs else np.nan,
        'HomeFormGoalsConceded': np.mean(h_gc) if h_gc else np.nan,
        'AwayFormPoints': sum(a_pts) if a_pts else np.nan,
        'AwayFormGoalsScored': np.mean(a_gs) if a_gs else np.nan,
        'AwayFormGoalsConceded': np.mean(a_gc) if a_gc else np.nan,
        'HomeManagerElo': h_mgr_elo, 'AwayManagerElo': a_mgr_elo,
        'HomeManagerKnown': int(home_mgr is not None), 'AwayManagerKnown': int(away_mgr is not None),
        'HomeDaysRest': h_days_rest, 'AwayDaysRest': a_days_rest,
        'HomeShortRest': int(h_days_rest <= 4), 'AwayShortRest': int(a_days_rest <= 4),
        'HomeAttack': h_attack, 'HomeDefence': h_defence, 'AwayAttack': a_attack, 'AwayDefence': a_defence,
        'EloCloseness': -abs(h_elo - a_elo),
        'AttackCloseness': -abs(h_attack - a_attack) if not (np.isnan(h_attack) or np.isnan(a_attack)) else np.nan,
    }


def predict_match_outcome(features, home_attack_raw, away_defence_raw, away_attack_raw, home_defence_raw, home_advantage):
    X_pred = pd.DataFrame([features])[feature_cols_simulation]
    proba = simulation_model.predict_proba(X_pred)[0]
    outcome = np.random.choice(['H', 'D', 'A'], p=proba)

    home_adv_log = np.log(1 + home_advantage)
    lambda_home = np.clip(np.exp(home_attack_raw + away_defence_raw + home_adv_log), 0.1, 5)
    lambda_away = np.clip(np.exp(away_attack_raw + home_defence_raw), 0.1, 5)

    home_goals, away_goals = None, None
    for _ in range(30):
        h_g = np.random.poisson(lambda_home)
        a_g = np.random.poisson(lambda_away)
        if outcome == 'H' and h_g > a_g: home_goals, away_goals = h_g, a_g; break
        if outcome == 'D' and h_g == a_g: home_goals, away_goals = h_g, a_g; break
        if outcome == 'A' and h_g < a_g: home_goals, away_goals = h_g, a_g; break
    if home_goals is None:
        if outcome == 'H': home_goals, away_goals = 1, 0
        elif outcome == 'A': home_goals, away_goals = 0, 1
        else: home_goals, away_goals = 1, 1
    return outcome, home_goals, away_goals


def update_simulation_state(home, away, match_date, home_goals, away_goals, sim_state, matchweek_number):
    """matchweek_number هنا دايماً من remaining_fixtures (موسم 2627 الحالي فقط) — فالتكيّف يسري صح تلقائياً"""
    adaptive_k = get_adaptive_k_factor(matchweek_number)

    if home not in sim_state['elo']: sim_state['elo'][home] = DEFAULT_ELO
    if away not in sim_state['elo']: sim_state['elo'][away] = DEFAULT_ELO
    new_h, new_a = update_elo_ratings(sim_state['elo'][home], sim_state['elo'][away], home_goals, away_goals,
                                        k_factor=adaptive_k, home_advantage=HOME_ADVANTAGE)
    sim_state['elo'][home], sim_state['elo'][away] = new_h, new_a

    home_mgr, away_mgr = current_managers.get(home), current_managers.get(away)
    if home_mgr and away_mgr:
        if home_mgr not in sim_state['manager_elo']: sim_state['manager_elo'][home_mgr] = init_manager_elo(home_mgr)
        if away_mgr not in sim_state['manager_elo']: sim_state['manager_elo'][away_mgr] = init_manager_elo(away_mgr)
        new_hm, new_am = update_elo_ratings(sim_state['manager_elo'][home_mgr], sim_state['manager_elo'][away_mgr],
                                              home_goals, away_goals, k_factor=adaptive_k, home_advantage=HOME_ADVANTAGE)
        sim_state['manager_elo'][home_mgr], sim_state['manager_elo'][away_mgr] = new_hm, new_am

    if home_goals > away_goals: h_pts, a_pts = 3, 0
    elif home_goals < away_goals: h_pts, a_pts = 0, 3
    else: h_pts, a_pts = 1, 1

    for team_name, pts, gs, gc in [(home, h_pts, home_goals, away_goals), (away, a_pts, away_goals, home_goals)]:
        for d in [sim_state['points_hist'], sim_state['gs_hist'], sim_state['gc_hist']]:
            if team_name not in d: d[team_name] = deque(maxlen=FORM_WINDOW)
        sim_state['points_hist'][team_name].append(pts)
        sim_state['gs_hist'][team_name].append(gs)
        sim_state['gc_hist'][team_name].append(gc)

    sim_state['last_date'][home] = match_date
    sim_state['last_date'][away] = match_date

print("✅ Part 4b — محرك المحاكاة جاهز (K-Factor تكيّفي مربوط صح بالجولة الفعلية).")

✅ Part 4b — محرك المحاكاة جاهز (K-Factor تكيّفي مربوط صح بالجولة الفعلية).


In [20]:
# ============================
# Part 5 — run_single_simulation + اختبار سريع
# ============================

def run_single_simulation(fixtures, base_state, seed=None):
    if seed is not None:
        np.random.seed(seed)
    sim_state = {k: copy.deepcopy(base_state[k]) for k in base_state}
    results = []
    for _, fixture in fixtures.iterrows():
        home, away, match_date = fixture['HomeTeam'], fixture['AwayTeam'], fixture['Date']
        matchweek_number = fixture['Round']
        features = build_simulation_features(home, away, match_date, sim_state)
        h_attack_raw, h_defence_raw = attack_dict.get(home, 0), defence_dict.get(home, 0)
        a_attack_raw, a_defence_raw = attack_dict.get(away, 0), defence_dict.get(away, 0)
        outcome, home_goals, away_goals = predict_match_outcome(
            features, h_attack_raw, a_defence_raw, a_attack_raw, h_defence_raw,
            params.get('home_advantage', 0.2)
        )
        update_simulation_state(home, away, match_date, home_goals, away_goals, sim_state, matchweek_number)
        results.append({'Date': match_date, 'HomeTeam': home, 'AwayTeam': away,
                         'HomeGoals': home_goals, 'AwayGoals': away_goals, 'Outcome': outcome})
    return pd.DataFrame(results)


def build_league_table(match_results, current_20_teams):
    table = {team: {'Points': 0, 'GF': 0, 'GA': 0, 'Played': 0, 'W': 0, 'D': 0, 'L': 0} for team in current_20_teams}
    for _, row in match_results.iterrows():
        home, away = row['HomeTeam'], row['AwayTeam']
        if home not in table or away not in table: continue
        hg = row['HomeGoals'] if 'HomeGoals' in row else row['FTHG']
        ag = row['AwayGoals'] if 'AwayGoals' in row else row['FTAG']
        table[home]['GF'] += hg; table[home]['GA'] += ag; table[home]['Played'] += 1
        table[away]['GF'] += ag; table[away]['GA'] += hg; table[away]['Played'] += 1
        if hg > ag: table[home]['Points'] += 3; table[home]['W'] += 1; table[away]['L'] += 1
        elif hg < ag: table[away]['Points'] += 3; table[away]['W'] += 1; table[home]['L'] += 1
        else:
            table[home]['Points'] += 1; table[away]['Points'] += 1
            table[home]['D'] += 1; table[away]['D'] += 1
    df_table = pd.DataFrame(table).T
    df_table['GD'] = df_table['GF'] - df_table['GA']
    df_table = df_table.sort_values(['Points', 'GD', 'GF'], ascending=False).reset_index()
    df_table.columns = ['Team', 'Points', 'GF', 'GA', 'Played', 'W', 'D', 'L', 'GD']
    df_table.index = df_table.index + 1
    return df_table[['Team', 'Played', 'W', 'D', 'L', 'GF', 'GA', 'GD', 'Points']]


# --- اختبار سريع: محاكاة واحدة ---
test_sim = run_single_simulation(remaining_fixtures, base_state, seed=42)
full_season_test = pd.concat([actual_played, test_sim[['HomeTeam','AwayTeam','HomeGoals','AwayGoals']]], ignore_index=True)
test_table = build_league_table(full_season_test, current_20_teams_list)

print(f"عدد المباريات المحاكاة: {len(test_sim)} (المتوقع: {len(remaining_fixtures)})")
print(f"\nجدول الترتيب (محاكاة واحدة فقط):")
print(test_table.to_string(index=False))
print(f"\nكل الفرق 38 مباراة؟ {(test_table['Played']==38).all()}")

عدد المباريات المحاكاة: 330 (المتوقع: 330)

جدول الترتيب (محاكاة واحدة فقط):
          Team  Played  W  D  L  GF  GA  GD  Points
      Man City      38 28  4  6  84  33  51      88
       Arsenal      38 23  6  9  69  33  36      75
     Brentford      38 19  9 10  62  47  15      66
     Tottenham      38 17 10 11  53  45   8      61
      Brighton      38 17  9 12  60  46  14      60
    Man United      38 18  6 14  55  51   4      60
       Everton      38 15 12 11  46  43   3      57
   Bournemouth      38 15 12 11  51  51   0      57
     Liverpool      38 16  8 14  64  53  11      56
       Chelsea      38 15  9 14  58  55   3      54
  Leeds United      38 14 11 13  45  51  -6      53
   Aston Villa      38 13 12 13  53  53   0      51
    Sunderland      38 14  9 15  46  47  -1      51
     Newcastle      38 14  8 16  56  63  -7      50
 Nott'm Forest      38 10 12 16  37  47 -10      42
     Hull City      38  8 14 16  36  47 -11      38
        Fulham      38  8 12 18  45  55

In [21]:
# ============================
# Part 6 — run_batch_simulations + محاكاة 10,000 كاملة
# ============================

def run_batch_simulations(fixtures, base_state, n_simulations, seed_start=0):
    all_states = [{k: copy.deepcopy(base_state[k]) for k in base_state} for _ in range(n_simulations)]
    all_results = [[] for _ in range(n_simulations)]
    np.random.seed(seed_start)
    home_advantage = params.get('home_advantage', 0.2)

    for _, fixture in fixtures.iterrows():
        home, away, match_date = fixture['HomeTeam'], fixture['AwayTeam'], fixture['Date']
        matchweek_number = fixture['Round']

        batch_features = [build_simulation_features(home, away, match_date, all_states[i]) for i in range(n_simulations)]
        X_batch = pd.DataFrame(batch_features)[feature_cols_simulation]
        proba_batch = simulation_model.predict_proba(X_batch)

        h_attack_raw, a_defence_raw = attack_dict.get(home, 0), defence_dict.get(away, 0)
        a_attack_raw, h_defence_raw = attack_dict.get(away, 0), defence_dict.get(home, 0)
        home_adv_log = np.log(1 + home_advantage)
        lambda_home = np.clip(np.exp(h_attack_raw + a_defence_raw + home_adv_log), 0.1, 5)
        lambda_away = np.clip(np.exp(a_attack_raw + h_defence_raw), 0.1, 5)

        for i in range(n_simulations):
            outcome = np.random.choice(['H', 'D', 'A'], p=proba_batch[i])
            home_goals, away_goals = None, None
            for _ in range(30):
                h_g, a_g = np.random.poisson(lambda_home), np.random.poisson(lambda_away)
                if outcome == 'H' and h_g > a_g: home_goals, away_goals = h_g, a_g; break
                if outcome == 'D' and h_g == a_g: home_goals, away_goals = h_g, a_g; break
                if outcome == 'A' and h_g < a_g: home_goals, away_goals = h_g, a_g; break
            if home_goals is None:
                home_goals, away_goals = (1,0) if outcome=='H' else (0,1) if outcome=='A' else (1,1)

            update_simulation_state(home, away, match_date, home_goals, away_goals, all_states[i], matchweek_number)
            all_results[i].append({'HomeTeam': home, 'AwayTeam': away, 'HomeGoals': home_goals, 'AwayGoals': away_goals})

    return [pd.DataFrame(r) for r in all_results]


N_SIMULATIONS = 10000
all_sim_results = run_batch_simulations(remaining_fixtures, base_state, n_simulations=N_SIMULATIONS, seed_start=42)

champion_counts = {t: 0 for t in current_20_teams_list}
top4_counts = {t: 0 for t in current_20_teams_list}
relegation_counts = {t: 0 for t in current_20_teams_list}
points_totals = {t: [] for t in current_20_teams_list}
position_totals = {t: [] for t in current_20_teams_list}

for sim_result in all_sim_results:
    full_season = pd.concat([actual_played, sim_result], ignore_index=True)
    table = build_league_table(full_season, current_20_teams_list)
    for position, row in enumerate(table.itertuples(), 1):
        if position == 1: champion_counts[row.Team] += 1
        if position <= 4: top4_counts[row.Team] += 1
        if position >= 18: relegation_counts[row.Team] += 1
        points_totals[row.Team].append(row.Points)
        position_totals[row.Team].append(position)

final_probabilities = pd.DataFrame({
    'Team': current_20_teams_list,
    'Champion_%': [champion_counts[t]/N_SIMULATIONS*100 for t in current_20_teams_list],
    'Top4_%': [top4_counts[t]/N_SIMULATIONS*100 for t in current_20_teams_list],
    'Relegation_%': [relegation_counts[t]/N_SIMULATIONS*100 for t in current_20_teams_list],
    'Avg_Points': [np.mean(points_totals[t]) for t in current_20_teams_list],
    'Avg_Position': [np.mean(position_totals[t]) for t in current_20_teams_list],
}).sort_values('Champion_%', ascending=False).reset_index(drop=True)
final_probabilities.index += 1

print(final_probabilities.to_string())

              Team  Champion_%  Top4_%  Relegation_%  Avg_Points  Avg_Position
1         Man City       68.06   99.30          0.00     86.8035        1.4334
2          Arsenal       25.89   95.79          0.00     80.0102        2.1734
3        Liverpool        4.82   71.27          0.06     69.5375        4.0467
4          Chelsea        0.31   26.48          0.86     58.8695        7.2262
5         Brighton        0.29   24.25          1.09     58.1915        7.3407
6       Man United        0.25   21.87          0.76     58.0680        7.3961
7        Newcastle        0.15   17.96          1.44     56.4437        8.1824
8        Brentford        0.13   16.21          1.88     55.5751        8.4285
9          Everton        0.05    6.29          5.15     50.4866       10.7188
10     Aston Villa        0.01    6.26          5.59     51.1330       10.3591
11    Leeds United        0.01    3.01          8.90     47.4840       12.0345
12  Crystal Palace        0.01    1.08         23.10

In [23]:
# ============================
# Part 7 (نهائية) — الحفظ الكامل (شامل كل نتائج الـ10,000 محاكاة + Confidence Bands)
# ============================

# --- Confidence Bands (P10/P50/P90) لكل فريق ---
confidence_bands = pd.DataFrame({
    'Team': current_20_teams_list,
    'Points_P10': [np.percentile(points_totals[t], 10) for t in current_20_teams_list],
    'Points_P50': [np.percentile(points_totals[t], 50) for t in current_20_teams_list],
    'Points_P90': [np.percentile(points_totals[t], 90) for t in current_20_teams_list],
    'Position_P10': [np.percentile(position_totals[t], 10) for t in current_20_teams_list],
    'Position_P50': [np.percentile(position_totals[t], 50) for t in current_20_teams_list],
    'Position_P90': [np.percentile(position_totals[t], 90) for t in current_20_teams_list],
})

final_project_save = {
    # البيانات الأساسية
    'df': df,
    'fixtures_df': fixtures_df,
    'actual_played': actual_played,
    'remaining_fixtures': remaining_fixtures,

    # حالة المحاكاة الحية
    'elo_ratings': elo_ratings,
    'manager_elo': manager_elo,
    'team_points_history': team_points_history,
    'team_goals_scored_history': team_goals_scored_history,
    'team_goals_conceded_history': team_goals_conceded_history,
    'team_xg_scored_history': team_xg_scored_history,
    'team_xg_conceded_history': team_xg_conceded_history,
    'last_match_date_full': last_match_date_full,
    'base_state': base_state,

    # بيانات المدربين
    'managers_df': managers_df,
    'current_managers': current_managers,
    'external_manager_experience': external_manager_experience,

    # Dixon-Coles
    'attack_dict': attack_dict,
    'defence_dict': defence_dict,
    'dc_model_params': params,
    'XI': XI,

    # XGBoost
    'simulation_model': simulation_model,
    'feature_cols_simulation': feature_cols_simulation,

    # ثوابت وتعريفات لازمة لإعادة التشغيل
    'master_name_map': master_name_map,
    'current_20_teams_list': current_20_teams_list,

    # نتائج المحاكاة الكاملة (كل الـ10,000 محاكاة خام)
    'all_sim_results': all_sim_results,
    'final_probabilities': final_probabilities,
    'confidence_bands': confidence_bands,
    'points_totals': points_totals,
    'position_totals': position_totals,
    'N_SIMULATIONS': N_SIMULATIONS,

    # معلومات توثيقية
    'accuracy_gate_result': {'accuracy': acc, 'log_loss': ll, 'baseline': 0.5066},
}

final_save_path = os.path.join(MODELS_PATH, 'FINAL_project_with_simulation.pkl')
temp_path = final_save_path + '.tmp'

with open(temp_path, 'wb') as f:
    pickle.dump(final_project_save, f)

os.replace(temp_path, final_save_path)

print(f"✅ تم الحفظ الآمن: {final_save_path}")
print(f"حجم الملف: {os.path.getsize(final_save_path) / 1024 / 1024:.1f} MB")

with open(final_save_path, 'rb') as f:
    verify_load = pickle.load(f)
print(f"✅ تأكيد: الملف بيتقرأ صح، فيه {len(verify_load)} عنصر.")

✅ تم الحفظ الآمن: /content/drive/MyDrive/EPL_Predictor_Project/models/FINAL_project_with_simulation.pkl
حجم الملف: 85.8 MB
✅ تأكيد: الملف بيتقرأ صح، فيه 31 عنصر.
